# 14 — LSAR Full-Pol Complex Covariance Reconstruction

Only runs if Module 13 detected full polarimetry. Reuses the Module 06 subset as-is — the HDF5 file stays on disk, and only the six upper-triangular covariance layers for that subset get pulled into memory.

Before anything gets decomposed, the three complex off-diagonal terms (HHHV, HHVV, HVVV) get a direct visual QC check — magnitude and phase, side by side — since a calibration artifact or residual transmit-gap contamination in one specific channel is otherwise invisible until it shows up, distorted, in a downstream H-A-α or Pauli/Freeman–Durden/Yamaguchi result several modules later.

In [ ]:
import numpy as np
from nisar_utils.config import load_config
from nisar_utils.polarimetric_io import load_detected_mode, get_authoritative_subset, load_branch_terms, FULL_POL_TERMS
from nisar_utils.polarimetry import covariance_hermitian_from_terms, hermitian_relative_error
cfg=load_config(); mode=load_detected_mode(cfg)
if mode['polarimetric_mode']!='LSAR_FULL_POL':
    print('SKIPPED: detected branch is',mode['polarimetric_mode'],'— Module 18 is the compact-pol branch when applicable.')
else:
    data,sp,guard,mode=load_branch_terms(cfg,FULL_POL_TERMS)
    C=covariance_hermitian_from_terms(data)
    err=hermitian_relative_error(C)
    print('Branch:',mode['polarimetric_mode'])
    print('Exact persisted subset:',sp,'shape=',C.shape[:-2])
    print('Terms loaded:',list(data))
    print('Estimated polarimetric RAM:',guard['estimated_gb'],'GB; safe=',guard['safe'])
    print('Median Hermitian relative error:',float(np.nanmedian(err)))
    print('95th percentile:',float(np.nanpercentile(err,95)))

In [ ]:
if mode['polarimetric_mode']=='LSAR_FULL_POL':
    print('Complex off-diagonal terms are retained as complex arrays:')
    for name in ('HHHV','HHVV','HVVV'):
        i,j={'HHHV':(0,1),'HHVV':(0,2),'HVVV':(1,2)}[name]
        a=C[...,i,j]
        print(name,'median magnitude=',float(np.nanmedian(np.abs(a))))

In [ ]:
import matplotlib.pyplot as plt
from nisar_utils.visualization import plot_complex_term

if mode['polarimetric_mode']=='LSAR_FULL_POL':
    for name in ('HHHV','HHVV','HVVV'):
        i,j={'HHHV':(0,1),'HHVV':(0,2),'HVVV':(1,2)}[name]
        a=C[...,i,j]
        fig, (ax_mag, ax_phase) = plt.subplots(1, 2, figsize=(12, 5))
        plot_complex_term(a, name, ax_mag=ax_mag, ax_phase=ax_phase)
        fig.suptitle(f'{name} -- QC before decomposition ({mode["polarimetric_frequency"]})')
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        print(name, 'valid (non-NaN) pixels:', int(np.count_nonzero(~np.isnan(np.abs(a)))),
              'of', a.size)